In [1]:
import torch
import torch.nn as nn

In [2]:
class Embeddings(nn.Module):
   def __init__(self, vocab_size, embed_dim, max_len=512):
       super(Embeddings, self).__init__()
       self.embedding = nn.Embedding(vocab_size, embed_dim)
       # Define a positional encoding as a learnable parameter to encode positional information.
       self.positional_encoding = nn.Parameter(torch.zeros(1, max_len, embed_dim))

   def forward(self, x):
       # Get the sequence length from the input tensor.
       seq_len = x.size(1)
       x = self.embedding(x)
       # Add positional encoding to the word embeddings, using only up to 'seq_len' positions.
       x += self.positional_encoding[:, :seq_len, :]
       return x


In [3]:
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(SelfAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        # Calculate the dimension for each attention head.
        self.head_dim = embed_dim // num_heads

        # Define linear layers for transforming input into queries(Q), keys(K), and values(V).
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)
        # Define a linear layer for transforming the concatenated attention outputs back into 'embed_dim'.
        self.out_linear = nn.Linear(embed_dim, embed_dim)
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)

        # Reshape Q, K, V for multi-head attention and transpose for easier computation.
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Calculate attention scores using dot-product and scale by the square root of 'head_dim' for stability.
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Apply softmax to get attention weights, normalizing over the last dimension.
        attention_weights = torch.softmax(scores, dim=-1)
        # Compute the weighted sum of values (V) using attention weights.
        attention_output = torch.matmul(attention_weights, V)
        # Reshape the output back into the original tensor shape and apply the final linear layer.
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        return self.out_linear(attention_output)


In [4]:
class AddNorm(nn.Module):
   def __init__(self, embed_dim):
       super(AddNorm, self).__init__()
       self.norm = nn.LayerNorm(embed_dim)

   def forward(self, x, sublayer_output):
       return self.norm(x + sublayer_output)

class FeedForward(nn.Module):
   def __init__(self, embed_dim, hidden_dim=2048):
       super(FeedForward, self).__init__()
       self.linear1 = nn.Linear(embed_dim, hidden_dim)
       self.relu = nn.ReLU()
       self.linear2 = nn.Linear(hidden_dim, embed_dim)

   def forward(self, x):
       return self.linear2(self.relu(self.linear1(x)))


In [5]:
class EncoderLayer(nn.Module):
   def __init__(self, embed_dim, num_heads, hidden_dim=2048):
       super(EncoderLayer, self).__init__()
       self.self_attention = SelfAttention(embed_dim, num_heads)
       self.add_norm1 = AddNorm(embed_dim)
       self.feed_forward = FeedForward(embed_dim, hidden_dim)
       self.add_norm2 = AddNorm(embed_dim)

   def forward(self, x):
       attention_output = self.self_attention(x)
       x = self.add_norm1(x, attention_output)
       ffn_output = self.feed_forward(x)
       x = self.add_norm2(x, ffn_output)
       return x


In [6]:
class TransformerEncoder(nn.Module):
   def __init__(self, vocab_size, embed_dim, num_heads, num_layers, hidden_dim=2048,max_len=512):
       super(TransformerEncoder, self).__init__()
       self.embedding = Embeddings(vocab_size, embed_dim, max_len)
       #Stack multiple EncoderLayer instances using ModuleList for the specified number of layers.
       self.encoder_layers = nn.ModuleList([
           EncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)
       ])

   def forward(self, x):
       x = self.embedding(x)
       # Sequentially pass the embeddings through each encoder layer.
       for layer in self.encoder_layers:
           x = layer(x)
       return x


In [7]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MaskedSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads

        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)

        self.out_linear = nn.Linear(embed_dim, embed_dim)
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Create a lower triangular mask to prevent attending to future tokens in the sequence.
        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)

        # Apply the mask, setting the scores of future positions to negative infinity.
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        return self.out_linear(attention_output)


In [8]:
class EncoderDecoderAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(EncoderDecoderAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads

        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)

        self.out_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, encoder_output):
        batch_size, seq_len, _ = x.size()
        # Get the sequence length of the encoder's output.
        enc_seq_len = encoder_output.size(1)

        # Compute Q from the decoder's input, and K, V from the encoder's output.
        Q = self.q_linear(x)
        K = self.k_linear(encoder_output)
        V = self.v_linear(encoder_output)

        # Reshape Q, K, V for multi-head attention and transpose for easier computation.
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, enc_seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, enc_seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention_weights = torch.softmax(scores, dim=-1)

        attention_output = torch.matmul(attention_weights, V)
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        return self.out_linear(attention_output)


In [9]:
class DecoderLayer(nn.Module):
   def __init__(self, embed_dim, num_heads, hidden_dim=2048):
       super(DecoderLayer, self).__init__()
       self.masked_self_attention = MaskedSelfAttention(embed_dim, num_heads)
       self.add_norm1 = AddNorm(embed_dim)
       self.encoder_decoder_attention = EncoderDecoderAttention(embed_dim, num_heads)
       self.add_norm2 = AddNorm(embed_dim)
       self.feed_forward = FeedForward(embed_dim, hidden_dim)
       self.add_norm3 = AddNorm(embed_dim)

   def forward(self, x, encoder_output):
       masked_attention_output = self.masked_self_attention(x)
       x = self.add_norm1(x, masked_attention_output)
       enc_dec_attention_output = self.encoder_decoder_attention(x, encoder_output)
       x = self.add_norm2(x, enc_dec_attention_output)
       ffn_output = self.feed_forward(x)
       x = self.add_norm3(x, ffn_output)
       return x


In [10]:
class TransformerDecoder(nn.Module):
   def __init__(self, vocab_size, embed_dim, num_heads, num_layers, hidden_dim=2048, max_len=512):
       super(TransformerDecoder, self).__init__()
       self.embedding = Embeddings(vocab_size, embed_dim, max_len)
       # Stack multiple DecoderLayer instances using ModuleList for the specified number of layers.
       self.decoder_layers = nn.ModuleList([
           DecoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)
       ])

   def forward(self, x, encoder_output):
       x = self.embedding(x)
       # Sequentially pass the embeddings through each decoder layer, using the encoder's output.
       for layer in self.decoder_layers:
           x = layer(x, encoder_output)
       return x


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Prepare translation data pairs (English and Spanish sentences).
data = [
   ["hello", "hola"],
   ["how are you", "cómo estás"],
   ["good morning", "buenos días"],
   ["thank you", "gracias"],
   ["see you later", "hasta luego"],
   ["good night", "buenas noches"],
   ["please", "por favor"],
   ["yes", "sí"],
   ["no", "no"],
   ["what is your name", "cuál es tu nombre"],
   ["my name is", "mi nombre es"],
]


cuda


In [12]:
# Define a simple tokenizer to convert words to indices and vice versa.
class SimpleTokenizer:
   def __init__(self):
       self.word2idx = {}
       self.idx2word = {}
       self.vocab_size = 0

   def build_vocab(self, sentences):
       # Build a vocabulary from the provided sentences.
       # Each word is mapped to a unique index, starting from 4 (0-3 reserved for special tokens).
       unique_words = set(word for sentence in sentences for word in sentence.split())
       self.word2idx = {word: idx + 4 for idx, word in enumerate(unique_words)}
       self.word2idx["<pad>"] = 0
       self.word2idx["<sos>"] = 1
       self.word2idx["<eos>"] = 2
       self.word2idx["<unk>"] = 3
       self.idx2word = {idx: word for word, idx in self.word2idx.items()}
       self.vocab_size = len(self.word2idx)

   def encode(self, sentence):
       # Convert a sentence into a list of token indices.
       return [self.word2idx.get(word, self.word2idx["<unk>"]) for word in sentence.split()]

   def decode(self, indices):
       # Convert a list of token indices back into a sentence.
       return " ".join([self.idx2word.get(idx, "<unk>") for idx in indices])


In [13]:
# Create tokenizers for English and Spanish languages.
english_tokenizer = SimpleTokenizer()
spanish_tokenizer = SimpleTokenizer()

# Extract English and Spanish sentences from the data and build vocabularies.
english_sentences = [pair[0] for pair in data]
spanish_sentences = [pair[1] for pair in data]

english_tokenizer.build_vocab(english_sentences)
spanish_tokenizer.build_vocab(spanish_sentences)

# Function to pad token sequences to a fixed length.
def pad_sequence(seq, max_len, pad_idx=0):
   # Add padding tokens (index 0) to reach the desired sequence length.
   return seq + [pad_idx] * (max_len - len(seq))


In [14]:
# Preprocess the data by tokenizing and padding sentences.
def preprocess_data(data, max_len):
   english_data = []
   spanish_data = []

   for eng, spa in data:
       # Encode sentences and add <sos> (1) and <eos> (2) tokens.
       eng_tokens = [1] + english_tokenizer.encode(eng) + [2]  # <sos> ... <eos>
       spa_tokens = [1] + spanish_tokenizer.encode(spa) + [2]  # <sos> ... <eos>

       # Pad sequences to the maximum length.
       eng_tokens = pad_sequence(eng_tokens, max_len)
       spa_tokens = pad_sequence(spa_tokens, max_len)

       english_data.append(eng_tokens)
       spanish_data.append(spa_tokens)

   # Convert the lists into PyTorch tensors.
   return torch.tensor(english_data), torch.tensor(spanish_data)

# Define the maximum sequence length for input data.
max_len = 10
english_tensor, spanish_tensor = preprocess_data(data, max_len)


In [15]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Define a dataset class for translation tasks.
class TranslationDataset(Dataset):
   def __init__(self, src_data, tgt_data):
       self.src_data = src_data
       self.tgt_data = tgt_data

   def __len__(self):
       # Return the number of data pairs.
       return len(self.src_data)

   def __getitem__(self, idx):
       # Return a source sentence, target input (without the last token),
   # and target output (without the first token).
       return self.src_data[idx], self.tgt_data[idx][:-1], self.tgt_data[idx][1:]

# Create a DataLoader to iterate through the dataset.
dataset = TranslationDataset(english_tensor, spanish_tensor)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [16]:
class Transformer(nn.Module):
   def __init__(self, src_vocab_size, tgt_vocab_size, embed_dim, num_heads, num_layers, hidden_dim, max_len):
       super(Transformer, self).__init__()
       # Initialize the encoder and decoder components.
       self.encoder = TransformerEncoder(src_vocab_size, embed_dim, num_heads, num_layers, hidden_dim, max_len)
       self.decoder = TransformerDecoder(tgt_vocab_size, embed_dim, num_heads, num_layers, hidden_dim, max_len)

       # Define an output linear layer that projects the decoder output to the target vocabulary size.
       self.output_layer = nn.Linear(embed_dim, tgt_vocab_size)
       self.device = device

   def forward(self, src, tgt):
       src = src.to(self.device)
       tgt = tgt.to(self.device)

       # Pass the source input through the encoder.
       encoder_output = self.encoder(src)

       # Pass the encoder output and target input through the decoder.
       decoder_output = self.decoder(tgt, encoder_output)

       # Project the decoder output to the target vocabulary size.
       return self.output_layer(decoder_output)


In [17]:
# Set the hyperparameters and instantiate the Transformer model.
embed_dim = 512
num_heads = 8
num_layers = 2
hidden_dim = 2048

model = Transformer(
   src_vocab_size=english_tokenizer.vocab_size,
   tgt_vocab_size=spanish_tokenizer.vocab_size,
   embed_dim=embed_dim,
   num_heads=num_heads,
   num_layers=num_layers,
   hidden_dim=hidden_dim,
   max_len=max_len
).to(device)

# Define the loss function and optimizer.
# Ignore padding tokens when calculating loss.
criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

MODEL_PATH = "transformer_translation_model.pth"


In [18]:
def train(model, dataloader, criterion, optimizer, num_epochs=10):
   for epoch in range(num_epochs):
       model.train()  # Set the model to training mode.
       epoch_loss = 0

       for src, tgt_input, tgt_output in dataloader:
           optimizer.zero_grad()  # Clear previous gradients.

           # Move input and output tensors to the appropriate device.
           src = src.to(device)
           tgt_input = tgt_input.to(device)
           tgt_output = tgt_output.to(device)

           # Pass the source and target input through the model.
           outputs = model(src, tgt_input)
           # Compute the loss between the model output and the target output.
           loss = criterion(outputs.view(-1, outputs.size(-1)), tgt_output.view(-1))
           loss.backward()  # Compute gradients.
           optimizer.step()  # Update model parameters.
           epoch_loss += loss.item()  # Accumulate the loss for the epoch.

       print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(dataloader):.4f}')

   # Save the trained model weights.
   torch.save(model.state_dict(), MODEL_PATH)
   print(f'Model saved to {MODEL_PATH}')


In [19]:
# Convert a test input sentence into a tensor that can be used by the model.
def tokenize_input(sentence, tokenizer, max_len):
   tokens = tokenizer.encode(sentence)
   tokens = [1] + tokens + [2]  # Add <sos> ... <eos> tokens.
   tokens = pad_sequence(tokens, max_len)
   return torch.tensor(tokens).unsqueeze(0)  # Add a batch dimension.

def test(model, input_sentence, english_tokenizer, spanish_tokenizer, max_len=10):
   model.eval()
   src_tensor = tokenize_input(input_sentence, english_tokenizer, max_len).to(device)

   with torch.no_grad():
       encoder_output = model.encoder(src_tensor) # Pass the source input through the encoder.

       tgt_tokens = torch.tensor([[1]]).to(device)  # Start decoding with the <sos> token.

       translated_tokens = []
       for _ in range(max_len):
           with torch.no_grad():
               # Decode step-by-step, passing the previous tokens to generate the next one.
               output = model.decoder(tgt_tokens, encoder_output)
               output = model.output_layer(output)
           # Get the index of the most likely next token.
           next_token = output[:, -1, :].argmax(1).item()
           if next_token == 2:  # Stop decoding if <eos> token is generated.
               break
           translated_tokens.append(next_token)
           # Add the predicted token to the sequence for the next step.
           tgt_tokens = torch.cat([tgt_tokens, torch.tensor([[next_token]]).to(device)], dim=1)

       # Decode the list of token indices into a translated sentence.
       translated_sentence = spanish_tokenizer.decode(translated_tokens)
       return translated_sentence


In [20]:

# Train the model using the training function.
train(model, dataloader, criterion, optimizer, num_epochs=10)

# Load the saved model weights for testing.
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
print(f'Model loaded from {MODEL_PATH}')

# Translate a test sentence using the trained model.
input_sentence = "good night"
translated_sentence = test(model, input_sentence, english_tokenizer, spanish_tokenizer, max_len=10)
print(f'Input: {input_sentence}')
print(f'Translation: {translated_sentence}')


Epoch 1/10, Loss: 3.0920
Epoch 2/10, Loss: 1.8905
Epoch 3/10, Loss: 1.3305
Epoch 4/10, Loss: 0.8840
Epoch 5/10, Loss: 0.7233
Epoch 6/10, Loss: 0.4346
Epoch 7/10, Loss: 0.2044
Epoch 8/10, Loss: 0.0996
Epoch 9/10, Loss: 0.0516
Epoch 10/10, Loss: 0.0291
Model saved to transformer_translation_model.pth
Model loaded from transformer_translation_model.pth
Input: good night
Translation: buenas noches
